## This notebook implements a complete PySpark pipeline for the IDS2018 dataset, covering data ingestion, preprocessing, feature selection, model training, threshold tuning, and metrics reporting.
## The focus of this run is to evaluate performance when deploying a single Spark worker configured with one CPU core, in order to benchmark accuracy, training time, and parallelism efficiency under minimal computational resources

# Importing Essential Libraries

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark import StorageLevel

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import time
from pathlib import Path


## PySpark Setup for IDS2018: Driver & Executor Core Scaling (with one core)

In [1]:
import os, sys
from pyspark.sql import SparkSession

# Make sure the port and URL match the one you used to start the master
MASTER_URL = "spark://127.0.0.1:7077"

# Ensure that both the Driver and Executors use the same running Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .master(MASTER_URL)
         .appName("IDS2018-Notebook")
         # Executor resources (based on Worker configuration: 4 cores, 8GB)
         .config("spark.executor.cores", "1")       # Use available worker cores
         .config("spark.executor.memory", "6g")     # Leave some margin for JVM/OS out of 8GB
         # Driver resources (adjust according to your machine's memory)
         .config("spark.driver.memory", "4g")
         # Useful settings
         .config("spark.sql.shuffle.partitions", "128")
         .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
         .getOrCreate())

print("Spark:", spark.version, "| master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark: 3.5.6 | master: spark://127.0.0.1:7077
Default parallelism: 2


## IDS2018 Data Pipeline: Ingest CSVs, Clean Features, and Create Binary Labels

In [ ]:
# Ingest CSVs, clean columns, create binary label, cast numerics

import os
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CSV_DIR = r"C:\Users\HP\bigdata\final spark IDS\data"
OUT_DIR = r"C:\Users\HP\bigdata\final spark IDS\reports"
os.makedirs(OUT_DIR, exist_ok=True)

# Read all CICFlowMeter CSVs
raw = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .option("multiLine", True)
       .option("escape", '"')
       .csv(str(Path(CSV_DIR) / "*.csv")))
raw = raw.dropna(how="all").toDF(*[c.strip() for c in raw.columns])

# Label cleaning
if "Label" in raw.columns:
    df = raw.filter(F.col("Label") != "Label")  # drop leaked header rows
    df = df.withColumn("Label_clean", F.lower(F.trim(F.col("Label"))))
else:
    df = raw.withColumn("Label_clean", F.lit(None).cast("string"))

# Binary label: benign -> 0, attack -> 1
df = df.withColumn(
    "label",
    F.when(F.col("Label_clean") == F.lit("benign"), F.lit(0)).otherwise(F.lit(1)).cast("double")
)

# Drop ID/time/IP columns that can leak information
drop_cols = [c for c in ["Flow ID","Timestamp","Src IP","Dst IP","Src Port","Dst Port","Label","Label_clean"] if c in df.columns]
df = df.drop(*drop_cols)

# Cast remaining features to double and sanitize NaN/Inf
for c, t in df.dtypes:
    if c != "label":
        df = df.withColumn(c, F.col(c).cast(DoubleType()))
for c, t in df.dtypes:
    if c != "label" and t in ("double","int","bigint","float"):
        df = df.withColumn(
            c,
            F.when(~F.isnan(F.col(c)) & ~F.col(c).isin(float("inf"), float("-inf")), F.col(c)).otherwise(F.lit(0.0))
        )

# Final numeric feature list
num_cols = [c for c, t in df.dtypes if c != "label" and t in ("double","int","bigint","float")]

df = df.persist()
print("Rows:", df.count(), "| Numeric columns:", len(num_cols))
df.groupBy("label").count().orderBy("label").show(truncate=False)


## Feature Selection using Random Forest Importance (Top-K)

In [3]:
# Feature selection via RandomForest importance (Top-K)

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
import pandas as pd

TOPK = 40
SAMPLE_ROWS = 300_000  # speed up importance estimation

assembler_all = VectorAssembler(inputCols=num_cols, outputCol="features_all")
data_all = assembler_all.transform(df.select(*num_cols, "label")).select("features_all", "label")

cnt = data_all.count()
frac = min(1.0, SAMPLE_ROWS / float(cnt))
data_s = data_all.sample(False, frac, seed=42) if frac < 1.0 else data_all

rf_tmp = RandomForestClassifier(featuresCol="features_all", labelCol="label",
                                numTrees=60, maxDepth=10, seed=42)
rf_tmp_m = rf_tmp.fit(data_s)

imp = pd.Series(rf_tmp_m.featureImportances.toArray(), index=num_cols).sort_values(ascending=False)
top_features = list(imp.head(TOPK).index)

print("Selected Top-K features:", len(top_features))
print(top_features[:20])  # quick view

# Save importance table
imp_df = imp.reset_index()
imp_df.columns = ["feature", "importance"]
imp_df.to_csv(os.path.join(OUT_DIR, "feature_importances_tmp.csv"), index=False, encoding="utf-8-sig")


Selected Top-K features: 40
['Fwd Seg Size Min', 'Fwd Pkt Len Max', 'Pkt Len Mean', 'Fwd Seg Size Avg', 'Pkt Len Max', 'Fwd Pkts/s', 'Fwd Header Len', 'Init Fwd Win Byts', 'Flow Pkts/s', 'Fwd IAT Max', 'Flow IAT Max', 'Pkt Len Var', 'Flow Duration', 'Flow IAT Min', 'Pkt Size Avg', 'Subflow Fwd Byts', 'Bwd Pkts/s', 'Flow IAT Mean', 'TotLen Fwd Pkts', 'Init Bwd Win Byts']


## Feature Assembly, Split, and Class Balancing (Undersampling)

In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Assemble top-K features
assembler = VectorAssembler(inputCols=top_features, outputCol="features")
data = assembler.transform(df).select("features", F.col("label").cast("double").alias("label"))

# Train/Test split
trainDF, testDF = data.randomSplit([0.8, 0.2], seed=42)
trainDF = trainDF.persist(StorageLevel.MEMORY_AND_DISK)
testDF  = testDF.persist(StorageLevel.MEMORY_AND_DISK)

print("Train rows:", trainDF.count(), "| Test rows:", testDF.count())

# Simple undersampling on training set to reduce class imbalance
ben = trainDF.filter("label = 0.0")
att = trainDF.filter("label = 1.0")
ben_n, att_n = ben.count(), att.count()

ratio = att_n / max(1, ben_n)   # keep all attacks; sample benigns down to similar scale
train_bal = ben.sample(False, ratio, seed=42).union(att).persist(StorageLevel.MEMORY_AND_DISK)

print("Balanced train rows:", train_bal.count(), "| benign:", ben_n, "| attack:", att_n, "| sample ratio:", round(ratio, 4))


Train rows: 2516916 | Test rows: 628808
Balanced train rows: 1573330 | benign: 1730732 | attack: 786184 | sample ratio: 0.4542


## Train RF & GBT on the balanced train set and evaluate on the test set

In [5]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

# Evaluators
e_bin = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
e_mc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# --- Random Forest ---
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=160, maxDepth=12, subsamplingRate=0.8, seed=42)

t0 = time.time()
rf_m = rf.fit(train_bal)
t1 = time.time()
rf_pred = rf_m.transform(testDF)
t2 = time.time()

rf_auc = e_bin.evaluate(rf_pred)
rf_acc = e_mc.evaluate(rf_pred, {e_mc.metricName: "accuracy"})
rf_f1  = e_mc.evaluate(rf_pred, {e_mc.metricName: "f1"})

print(f"[RF] AUC={rf_auc:.4f} | Acc={rf_acc:.4f} | F1={rf_f1:.4f} | fit_s={t1-t0:.1f} | pred_s={t2-t1:.2f}")

# --- GBT ---
gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxIter=100, maxDepth=8, stepSize=0.1, seed=42)

t0 = time.time()
gbt_m = gbt.fit(train_bal)
t1 = time.time()
gbt_pred = gbt_m.transform(testDF)
t2 = time.time()

gbt_auc = e_bin.evaluate(gbt_pred)
gbt_acc = e_mc.evaluate(gbt_pred, {e_mc.metricName: "accuracy"})
gbt_f1  = e_mc.evaluate(gbt_pred, {e_mc.metricName: "f1"})

print(f"[GBT] AUC={gbt_auc:.4f} | Acc={gbt_acc:.4f} | F1={gbt_f1:.4f} | fit_s={t1-t0:.1f} | pred_s={t2-t1:.2f}")


[RF] AUC=1.0000 | Acc=0.9997 | F1=0.9997 | fit_s=626.3 | pred_s=0.25
[GBT] AUC=1.0000 | Acc=0.9999 | F1=0.9999 | fit_s=660.3 | pred_s=0.40


## Threshold Tuning and Confusion Matrix Evaluation (RF & GBT)

In [6]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

def ensure_score_column(pred_df):
    # ensure a probability-like score p(class=1)
    if "probability" in pred_df.columns:
        return pred_df.withColumn("p1", vector_to_array(F.col("probability"))[1])
    elif "rawPrediction" in pred_df.columns:
        rp1 = vector_to_array(F.col("rawPrediction"))[1]
        return pred_df.withColumn("p1", 1 / (1 + F.exp(-rp1)))
    else:
        raise ValueError("Missing 'probability' or 'rawPrediction'.")

def confusion_metrics_df(df_with_pred):
    agg = df_with_pred.agg(
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==0), 1).otherwise(0)).alias("tn"),
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==0), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==1), 1).otherwise(0)).alias("tp")
    ).collect()[0]
    tn, fp, fn, tp = float(agg["tn"]), float(agg["fp"]), float(agg["fn"]), float(agg["tp"])
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    specificity = tn / (tn + fp + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    fpr = fp / (fp + tn + 1e-12)
    fnr = fn / (fn + tp + 1e-12)
    return tn, fp, fn, tp, precision, recall, specificity, f1, fpr, fnr

def tune_threshold_safe(pred_df, name="Model", min_precision=0.2):
    scored = ensure_score_column(pred_df).select("p1", "label")
    thresholds = [x/100 for x in range(10, 90, 2)]  # 0.10 .. 0.88
    best = None
    best_key = (-1, -1, -1)  # (specificity, recall, f1)

    for th in thresholds:
        pr = scored.withColumn("pred", (F.col("p1") >= F.lit(th)).cast("double"))
        tn, fp, fn, tp, prec, rec, spec, f1, _, _ = confusion_metrics_df(pr)
        if prec >= min_precision:
            key = (spec, rec, f1)
            if key > best_key:
                best_key = key
                best = (th, prec, rec, spec, f1)

    if best is None:  # fallback to best F1 over all thresholds
        best_f1 = -1
        for th in thresholds:
            pr = scored.withColumn("pred", (F.col("p1") >= F.lit(th)).cast("double"))
            _, _, _, _, prec, rec, spec, f1, _, _ = confusion_metrics_df(pr)
            if f1 > best_f1:
                best_f1 = f1
                best = (th, prec, rec, spec, f1)

    th, prec, rec, spec, f1 = best
    print(f"[{name}] best threshold={th:.2f} | Precision={prec:.3f} | Recall={rec:.3f} | Specificity={spec:.3f} | F1={f1:.3f}")
    return best

def apply_threshold(pred_df, th):
    scored = ensure_score_column(pred_df)
    return scored.withColumn("prediction", (F.col("p1") >= F.lit(th)).cast("double"))

# Tune thresholds
best_rf  = tune_threshold_safe(rf_pred,  "RF",  min_precision=0.2)
best_gbt = tune_threshold_safe(gbt_pred, "GBT", min_precision=0.2)

# Apply and report confusion matrices + FPR/FNR
for name, pred_df, best in [
    ("RF_tuned",  rf_pred,  best_rf),
    ("GBT_tuned", gbt_pred, best_gbt),
]:
    tuned = apply_threshold(pred_df, best[0])
    tn, fp, fn, tp, prec, rec, spec, f1, fpr, fnr = confusion_metrics_df(
        tuned.select("label","prediction").withColumnRenamed("prediction","pred")
    )
    print(f"[{name}] CM: [[TN={tn:.0f} FP={fp:.0f}] [FN={fn:.0f} TP={tp:.0f}]] "
          f"| Precision={prec:.3f} | Recall={rec:.3f} | Specificity={spec:.3f} | F1={f1:.3f} "
          f"| FPR={fpr:.6f} | FNR={fnr:.6f}")


[RF] best threshold=0.82 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000
[GBT] best threshold=0.86 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000
[RF_tuned] CM: [[TN=431879 FP=0] [FN=42 TP=196887]] | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | FPR=0.000000 | FNR=0.000213
[GBT_tuned] CM: [[TN=431843 FP=36] [FN=40 TP=196889]] | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | FPR=0.000083 | FNR=0.000203


## Tuned Predictions and Metrics Export (Worker Run)

#### Rebuilds predictions from the trained RF and GBT models, retunes thresholds, and collects performance metrics (Precision, Recall, F1, FPR, FNR).
#### Saves the results into a CSV file along with runtime details (Master, Parallelism, Timestamp) for documentation and comparison.

In [7]:
import os, time, json
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# --- helpers (define if missing) ---
if 'ensure_score_column' not in globals():
    def ensure_score_column(pred_df):
        if 'probability' in pred_df.columns:
            return pred_df.withColumn('p1', vector_to_array(F.col('probability'))[1])
        rp1 = vector_to_array(F.col('rawPrediction'))[1]
        return pred_df.withColumn('p1', 1 / (1 + F.exp(-rp1)))

if 'apply_threshold' not in globals():
    def apply_threshold(pred_df, th):
        scored = ensure_score_column(pred_df)
        return scored.withColumn('prediction', (F.col('p1') >= F.lit(float(th))).cast('double'))

if 'collect_metrics' not in globals():
    def collect_metrics(df, name, threshold):
        row = df.agg(
            F.sum(F.when((F.col("label")==0) & (F.col("prediction")==0), 1).otherwise(0)).alias("tn"),
            F.sum(F.when((F.col("label")==0) & (F.col("prediction")==1), 1).otherwise(0)).alias("fp"),
            F.sum(F.when((F.col("label")==1) & (F.col("prediction")==0), 1).otherwise(0)).alias("fn"),
            F.sum(F.when((F.col("label")==1) & (F.col("prediction")==1), 1).otherwise(0)).alias("tp")
        ).collect()[0]
        tn, fp, fn, tp = float(row["tn"]), float(row["fp"]), float(row["fn"]), float(row["tp"])
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        f1   = 2*prec*rec / (prec + rec + 1e-12)
        fpr  = fp / (fp + tn + 1e-12)
        fnr  = fn / (fn + tp + 1e-12)
        return {
            "Algorithm": name,
            "Threshold": round(float(threshold), 4),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "Precision": prec, "Recall": rec, "F1": f1,
            "FPR": fpr, "FNR": fnr
        }

# --- rebuild predictions from trained models ---
rf_pred  = rf_m.transform(testDF)
gbt_pred = gbt_m.transform(testDF)

# --- retune thresholds (works whether function returns tuple or scalar) ---
best_rf  = tune_threshold_safe(rf_pred,  "RF",  min_precision=0.2)
best_gbt = tune_threshold_safe(gbt_pred, "GBT", min_precision=0.2)
th_rf  = best_rf[0]  if isinstance(best_rf,  (list, tuple)) else best_rf
th_gbt = best_gbt[0] if isinstance(best_gbt, (list, tuple)) else best_gbt

# --- apply thresholds to get tuned predictions ---
rf_tuned  = apply_threshold(rf_pred,  th_rf)
gbt_tuned = apply_threshold(gbt_pred, th_gbt)

# --- build & save metrics ---
REPORTS = Path(r"C:\Users\HP\bigdata\sparkk2\reports")
rows = [
    collect_metrics(rf_tuned.select("label","prediction"),  "Random Forest",          th_rf),
    collect_metrics(gbt_tuned.select("label","prediction"), "Gradient Boosted Tree",  th_gbt),
]
for r in rows:
    r.update({
        "Master": spark.sparkContext.master,
        "DefaultParallelism": spark.sparkContext.defaultParallelism,
        "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    })

df_metrics = pd.DataFrame(rows)
metrics_path = REPORTS / "worker_run_metrics_2core.csv"
df_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
print("Saved metrics CSV ->", metrics_path)
display(df_metrics)


[RF] best threshold=0.82 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000
[GBT] best threshold=0.86 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000
Saved metrics CSV -> C:\Users\HP\bigdata\sparkk2\reports\worker_run_metrics.csv


,Algorithm,Threshold,TN,FP,FN,TP,Precision,Recall,F1,FPR,FNR,Master,DefaultParallelism,Timestamp
0,Random Forest,0.82,431879,0,42,196887,1.000000,0.999787,0.999893,0.000000,0.000213,spark://127.0.0.1:7077,2,2025-09-24 18:53:50
1,Gradient Boosted Tree,0.86,431843,36,40,196889,0.999817,0.999797,0.999807,0.000083,0.000203,spark://127.0.0.1:7077,2,2025-09-24 18:53:50
